# MLflow로 Iris 학습과 test 평가 기록하기

train 데이터로 모델을 학습하고, 분리한 test 데이터의 정확도로 체크포인트를 비교합니다. train 정확도는 보조 지표로 별도 기록합니다.

여기서는 test 지표로 여러 체크포인트를 비교하므로 최종 선택에 한 번만 사용하는 독립 test 평가는 아닙니다. 엄격한 모델 선택 실험에서는 validation 집합과 최종 test 집합을 추가로 분리해야 합니다.


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [ ]:
import mlflow
import mlflow.pytorch


In [ ]:
def prepare_data(df):
    X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
    y = torch.tensor(df.iloc[:, -1].values, dtype=torch.long)
    return X, y

In [ ]:
def compute_accuracy(model, X, y):
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            predicted = model(X).argmax(dim=1)
            return (predicted == y).float().mean().item()
    finally:
        model.train(was_training)


In [ ]:
class IrisClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [ ]:
iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df["target"] = iris.target

In [ ]:
train_df, test_df = train_test_split(
    iris_df, test_size=0.2, random_state=42, stratify=iris_df["target"]
)


In [ ]:
train_dataset = mlflow.data.from_pandas(train_df, name="train", targets="target")
test_dataset = mlflow.data.from_pandas(test_df, name="test", targets="target")
X_train, y_train = prepare_data(train_df)
X_test, y_test = prepare_data(test_df)


In [ ]:
torch.manual_seed(42)
input_size = X_train.shape[1]
hidden_size = 16
output_size = len(iris.target_names)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)
scripted_model = IrisClassifier(input_size, hidden_size, output_size).to(device)
scripted_model = torch.jit.script(scripted_model)


In [ ]:
with mlflow.start_run() as run:
    mlflow.log_input(train_dataset, context="training")
    mlflow.log_input(test_dataset, context="evaluation")
    mlflow.log_params({
        "n_linear_layers": 2,
        "hidden_size": hidden_size,
        "activation": "ReLU",
        "criterion": "CrossEntropyLoss",
        "optimizer": "Adam",
        "learning_rate": 0.01,
        "split_seed": 42,
        "model_seed": 42,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
    })
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(scripted_model.parameters(), lr=0.01)

    for epoch in range(101):
        scripted_model.train()
        out = scripted_model(X_train)
        loss = criterion(out, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            train_accuracy = compute_accuracy(scripted_model, X_train, y_train)
            test_accuracy = compute_accuracy(scripted_model, X_test, y_test)
            model_info = mlflow.pytorch.log_model(
                pytorch_model=scripted_model,
                name=f"torch-iris-{epoch}",
                step=epoch,
                serialization_format="pickle",
                input_example=X_train[:5].cpu().float().numpy(),
                pip_requirements=[f"torch=={torch.__version__.split('+')[0]}", "numpy"],
            )
            mlflow.log_metric(
                key="train_accuracy", value=train_accuracy, step=epoch,
                model_id=model_info.model_id, dataset=train_dataset,
            )
            mlflow.log_metric(
                key="test_accuracy", value=test_accuracy, step=epoch,
                model_id=model_info.model_id, dataset=test_dataset,
            )


In [ ]:
ranked_checkpoints = mlflow.search_logged_models(
    experiment_ids=[run.info.experiment_id],
    filter_string=f"source_run_id='{run.info.run_id}'",
    order_by=[{"field_name": "metrics.test_accuracy", "ascending": False}],
    output_format="list",
)


In [ ]:
best_checkpoint = ranked_checkpoints[0]
print(f"Highest test accuracy: {best_checkpoint.name}")
print([metric for metric in best_checkpoint.metrics if metric.key == "test_accuracy"])


In [ ]:
worst_checkpoint = ranked_checkpoints[-1]
print(f"Lowest test accuracy: {worst_checkpoint.name}")
print([metric for metric in worst_checkpoint.metrics if metric.key == "test_accuracy"])
